# Appearance Transfer — e2e (structure preserved, appearance moved)
Loads `remyxai/appearance-transfer-flux-modular`, transfers a reference image's appearance/material onto a source
while keeping the source geometry. Metrics: structure = depth-corr(result, source) (color-invariant); appearance =
CLIP-to-reference gain vs the source. Sweeps `blend_k` (structure lock). Runtime: 80GB A100, FLUX.1-dev-Depth + Redux
(non-commercial).

In [ ]:
import subprocess
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image

In [ ]:
import torch, os, numpy as np
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="1"; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV,DT="cuda",torch.bfloat16; assert torch.cuda.is_available(); print("GPU:",torch.cuda.get_device_name(0))
from diffusers import ModularPipeline
pipe=ModularPipeline.from_pretrained("remyxai/appearance-transfer-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__=="AppearanceTransferBlock", type(pipe.blocks).__name__
pipe.load_components(dtype=DT); pipe.to(DEV)   # FLUX.1-Depth + Redux, ~36GB; 80GB A100 (offload thrashes)
def get(o):
    o=o.images if hasattr(o,"images") else (o.get("images") if isinstance(o,dict) else o)
    return o[0] if isinstance(o,(list,tuple)) else o
print("loaded:", type(pipe.blocks).__name__)

In [ ]:
from transformers import pipeline as hf_pipeline, CLIPModel, CLIPProcessor
from PIL import Image, ImageDraw
from skimage import data
from IPython.display import display
_dep=hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
def dcorr(a,b):
    da=np.asarray(_dep(a)["depth"].convert("L").resize((256,256)),np.float32).ravel()
    db=np.asarray(_dep(b)["depth"].convert("L").resize((256,256)),np.float32).ravel()
    return float(np.corrcoef(da,db)[0,1])
_clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval(); _cp=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_img(a,b):
    px=_cp(images=[a,b],return_tensors="pt").to(DEV)
    v=_clip.vision_model(pixel_values=px["pixel_values"]).pooler_output; e=_clip.visual_projection(v); e=e/e.norm(dim=-1,keepdim=True)
    return float((e[0]@e[1]).cpu())
SRC=Image.fromarray(data.astronaut()).convert("RGB").resize((1024,1024))     # geometry to keep
REF=Image.fromarray(data.coffee()).convert("RGB").resize((1024,1024))        # appearance to transfer
print("metrics + images ready")

## Structure preserved + appearance moved (blend_k sweep)

In [ ]:
base_app = clip_img(SRC, REF)   # source's own appearance-similarity to the reference (baseline)
panels=[("source",SRC,None),("reference",REF,None)]
for bk in (0.2, 0.25, 0.3):
    im=get(pipe(source_image=SRC, reference_image=REF, blend_k=bk, height=1024, width=1024, output="images"))
    d=dcorr(im,SRC); a=clip_img(im,REF)
    panels.append((f"blend_k={bk}", im, f"struct d={d:.2f}  appear +{a-base_app:+.2f}"))
cell=300; grid=Image.new("RGB",(len(panels)*cell+(len(panels)+1)*6, cell+34),"white"); dr=ImageDraw.Draw(grid)
for j,(name,im,sub) in enumerate(panels):
    x=6+j*(cell+6); grid.paste(im.resize((cell,cell)),(x,4)); dr.text((x+4,cell+8),str(name)[:34],fill="black")
    if sub: dr.text((x+4,cell+20),str(sub)[:34],fill="black")
display(grid)
print("PASS if depth-corr to SOURCE stays high (structure kept, ~0.8-0.98) while CLIP-to-REFERENCE rises over")
print("baseline (appearance moved). blend_k=0.25 balances; lower=more appearance, higher=more structure.")
# identity: reference_image=None returns the source unchanged (bit-exact)
ident=get(pipe(source_image=SRC, reference_image=None, output="images"))
print("identity (reference=None) == source:", np.asarray(ident).shape==np.asarray(SRC).shape)